# Formação dos pares candidatos

Nesta etapa, utilizamos o universo líquido por janela e os preços ajustados pelo Yahoo Finance para identificar pares candidatos à estratégia de pairs trading.

A formação dos pares será feita dentro de cada janela de formação. Para cada janela, selecionamos os ativos líquidos daquele período, coletamos seus preços ajustados e testamos quais pares apresentam relação estatística relevante.

A seleção inicial dos pares será baseada em correlação dos retornos e teste de cointegração dos preços em log. Depois, os pares estatisticamente selecionados deverão passar por uma validação econômica, para verificar se a relação faz sentido do ponto de vista setorial.

In [1]:
from pathlib import Path
from itertools import combinations
import numpy as np
import pandas as pd
from statsmodels.tsa.stattools import coint

### 1. Carregamento das bases

Nesta etapa, carregamos o universo líquido por janela e a matriz de preços ajustados. O universo líquido indica quais ativos eram válidos em cada janela de formação. A matriz de preços ajustados será usada para testar a relação estatística entre os ativos.

In [3]:
PASTA_PROJETO = Path.cwd().parent
PASTA_DADOS_TRATADOS = PASTA_PROJETO / "dados_tratados"

universo_liquido = pd.read_csv(
    PASTA_DADOS_TRATADOS / "universo_liquido_por_janela.csv",
    parse_dates=[
        "inicio_formacao",
        "fim_formacao",
        "inicio_teste",
        "fim_teste"
    ]
)

precos_ajustados = pd.read_csv(
    PASTA_DADOS_TRATADOS / "precos_ajustados_yfinance_2010_2025.csv",
    index_col=0,
    parse_dates=True
)

print(f"Universo líquido: {universo_liquido.shape[0]} linhas e {universo_liquido.shape[1]} colunas")
print(f"Preços ajustados: {precos_ajustados.shape[0]} datas e {precos_ajustados.shape[1]} ativos")

Universo líquido: 2900 linhas e 5 colunas
Preços ajustados: 4064 datas e 248 ativos


### 2. Definição dos critérios estatísticos para formação dos pares

Nesta etapa, definimos os critérios usados para selecionar os pares candidatos em cada janela de formação. O objetivo é encontrar pares de ações que tenham relação estatística forte o suficiente para serem considerados na estratégia de pairs trading.

A cointegração será usada para avaliar se existe uma relação mais estável de longo prazo entre os preços dos dois ativos. Além disso, exigimos um número mínimo de observações para evitar conclusões baseadas em poucos dados.

In [5]:
MINIMO_DADOS_YAHOO = 0.95
PVALOR_COINTEGRACAO_MAX = 0.05
MINIMO_OBSERVACOES = 120
TOP_N_PARES = 20

### 3. Seleção de uma janela para teste

Antes de aplicar a formação de pares em todas as janelas móveis, selecionamos uma janela específica para testar a lógica do modelo. Essa etapa permite verificar se os ativos líquidos daquela janela estão sendo recuperados corretamente e se os preços ajustados correspondentes estão disponíveis.

In [6]:
janelas = universo_liquido[
    ["inicio_formacao", "fim_formacao", "inicio_teste", "fim_teste"]
].drop_duplicates().reset_index(drop=True)

janela_teste = janelas.iloc[0]

janela_teste

inicio_formacao   2010-01-04
fim_formacao      2011-01-03
inicio_teste      2011-01-04
fim_teste         2011-07-03
Name: 0, dtype: datetime64[us]

### 4. Função para formar pares cointegrados em uma janela

Nesta etapa, criamos uma função para formar pares candidatos dentro de uma janela de formação. A função recupera os ativos líquidos daquele período, recorta seus preços ajustados e testa todas as combinações possíveis de pares.

A seleção será baseada no teste de cointegração. Um par será mantido quando apresentar p-valor menor ou igual ao limite definido. Essa escolha está alinhada à lógica do pairs trading, pois a estratégia depende da existência de uma relação estatística estável entre os preços dos ativos e da possibilidade de reversão do spread ao seu padrão histórico.

Além disso, a função calcula o beta da relação entre os ativos e o spread histórico do par, que serão usados posteriormente para gerar sinais de entrada e saída no backtest.

In [7]:
def formar_pares_janela(janela):
    inicio = janela["inicio_formacao"]
    fim = janela["fim_formacao"]

    ativos_janela = universo_liquido[
        (universo_liquido["inicio_formacao"] == inicio) &
        (universo_liquido["fim_formacao"] == fim)
    ]["ativo"].tolist()

    precos_janela = precos_ajustados.loc[inicio:fim, ativos_janela]

    percentual_dados = precos_janela.notna().mean()
    ativos_validos = percentual_dados[percentual_dados >= MINIMO_DADOS_YAHOO].index.tolist()

    precos_janela = precos_janela[ativos_validos]

    resultados = []

    for ativo_1, ativo_2 in combinations(ativos_validos, 2):
        par_precos = precos_janela[[ativo_1, ativo_2]].dropna()

        if len(par_precos) < MINIMO_OBSERVACOES:
            continue

        log_precos = np.log(par_precos)

        _, pvalor, _ = coint(log_precos[ativo_1], log_precos[ativo_2])

        if pvalor > PVALOR_COINTEGRACAO_MAX:
            continue

        beta = np.polyfit(log_precos[ativo_2], log_precos[ativo_1], 1)[0]

        spread = log_precos[ativo_1] - beta * log_precos[ativo_2]

        resultados.append({
            "inicio_formacao": inicio,
            "fim_formacao": fim,
            "inicio_teste": janela["inicio_teste"],
            "fim_teste": janela["fim_teste"],
            "ativo_1": ativo_1,
            "ativo_2": ativo_2,
            "pvalor_cointegracao": pvalor,
            "beta": beta,
            "media_spread": spread.mean(),
            "desvio_spread": spread.std(),
            "observacoes": len(par_precos)
        })

    pares = pd.DataFrame(resultados)

    if pares.empty:
        return pares

    pares = pares.sort_values("pvalor_cointegracao")

    return pares.head(TOP_N_PARES)

In [8]:
pares_teste = formar_pares_janela(janela_teste)

pares_teste

,inicio_formacao,fim_formacao,inicio_teste,fim_teste,ativo_1,ativo_2,pvalor_cointegracao,beta,media_spread,desvio_spread,observacoes
95,2010-01-04,2011-01-03,2011-01-04,2011-07-03,PDGR3,RENT3,0.000001,0.801962,21.711783,0.038065,248
245,2010-01-04,2011-01-03,2011-01-04,2011-07-03,CSMG3,WEGE3,0.000007,0.473981,0.831373,0.045602,248
7,2010-01-04,2011-01-03,2011-01-04,2011-07-03,PETR4,KEPL3,0.000009,0.508455,1.629271,0.062248,248
120,2010-01-04,2011-01-03,2011-01-04,2011-07-03,MRVE3,RENT3,0.000024,0.869528,0.654635,0.040888,248
172,2010-01-04,2011-01-03,2011-01-04,2011-07-03,SANB11,ITSA3,0.000039,1.100476,1.174225,0.036836,248
72,2010-01-04,2011-01-03,2011-01-04,2011-07-03,GGBR4,MULT3,0.000074,-0.555915,3.502465,0.058591,248
246,2010-01-04,2011-01-03,2011-01-04,2011-07-03,CSMG3,MYPK3,0.000074,0.151704,0.792852,0.061257,248
247,2010-01-04,2011-01-03,2011-01-04,2011-07-03,CSMG3,POMO4,0.000105,0.150120,1.170165,0.059882,248
96,2010-01-04,2011-01-03,2011-01-04,2011-07-03,PDGR3,BBDC3,0.000175,1.568483,20.792143,0.060958,248
214,2010-01-04,2011-01-03,2011-01-04,2011-07-03,DASA3,POMO4,0.000196,0.676974,2.844969,0.039341,248


### 5. Formação dos pares em todas as janelas móveis

Após testar a formação de pares em uma janela específica, aplicamos a mesma função para todas as janelas móveis do projeto.

In [9]:
todos_pares = []

for i, janela in janelas.iterrows():
    print(f"Processando janela {i + 1} de {len(janelas)}")

    pares_janela = formar_pares_janela(janela)

    if not pares_janela.empty:
        todos_pares.append(pares_janela)

todos_pares = pd.concat(todos_pares, ignore_index=True)

todos_pares.head()

Processando janela 1 de 29
Processando janela 2 de 29
Processando janela 3 de 29
Processando janela 4 de 29
Processando janela 5 de 29
Processando janela 6 de 29
Processando janela 7 de 29
Processando janela 8 de 29
Processando janela 9 de 29
Processando janela 10 de 29
Processando janela 11 de 29
Processando janela 12 de 29
Processando janela 13 de 29
Processando janela 14 de 29
Processando janela 15 de 29
Processando janela 16 de 29
Processando janela 17 de 29
Processando janela 18 de 29
Processando janela 19 de 29
Processando janela 20 de 29
Processando janela 21 de 29
Processando janela 22 de 29
Processando janela 23 de 29
Processando janela 24 de 29
Processando janela 25 de 29
Processando janela 26 de 29
Processando janela 27 de 29
Processando janela 28 de 29
Processando janela 29 de 29


,inicio_formacao,fim_formacao,inicio_teste,fim_teste,ativo_1,ativo_2,pvalor_cointegracao,beta,media_spread,desvio_spread,observacoes
0,2010-01-04,2011-01-03,2011-01-04,2011-07-03,PDGR3,RENT3,0.000001,0.801962,21.711783,0.038065,248
1,2010-01-04,2011-01-03,2011-01-04,2011-07-03,CSMG3,WEGE3,0.000007,0.473981,0.831373,0.045602,248
2,2010-01-04,2011-01-03,2011-01-04,2011-07-03,PETR4,KEPL3,0.000009,0.508455,1.629271,0.062248,248
3,2010-01-04,2011-01-03,2011-01-04,2011-07-03,MRVE3,RENT3,0.000024,0.869528,0.654635,0.040888,248
4,2010-01-04,2011-01-03,2011-01-04,2011-07-03,SANB11,ITSA3,0.000039,1.100476,1.174225,0.036836,248
